# Robust CIFAR-10 classification with 500 images
Seeds: **split seed = 42**, **corruption seed = 1234**, **training seeds = 0, 1, 2**. Train from scratch, Inception-style CNN, <= 1M params, on-the-fly augmentation, <= 60 epochs.
All hyperparameters are chosen on the 100-image validation split only. Test sets are used once at the end.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, random, copy, time
import matplotlib.pyplot as plt, pandas as pd
import torchvision, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFilter
from sklearn.metrics import confusion_matrix
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS=[0,1,2]; SPLIT_SEED=42; CORRUPT_SEED=1234; EPOCHS=60
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print(dev)

## Data: train_small (400 train / 100 val), clean_test, shifted_test

In [ ]:
tr=torchvision.datasets.CIFAR10('./data',train=True,download=True)
te=torchvision.datasets.CIFAR10('./data',train=False,download=True)
classes=tr.classes
y_all=np.array(tr.targets); rng=np.random.RandomState(SPLIT_SEED)
tr_idx,va_idx=[],[]
for c in range(10):
    idx=rng.permutation(np.where(y_all==c)[0])[:50]       # 50 per class = 500
    tr_idx+=list(idx[:40]); va_idx+=list(idx[40:])         # 40 train + 10 val per class
Xtr,ytr=tr.data[tr_idx],y_all[tr_idx]; Xva,yva=tr.data[va_idx],y_all[va_idx]
Xte,yte=te.data,np.array(te.targets)
print(Xtr.shape,Xva.shape,Xte.shape)

# ---------- corruption pipeline (fixed + seeded) ----------
def hidden_corruption(im,rng):
    # STAND-IN for the hidden corruption: random grayscale or channel swap.
    # Replace with the official implementation once provided.
    if rng.rand()<0.5: return im.convert('L').convert('RGB')
    a=np.asarray(im); return Image.fromarray(a[:,:,rng.permutation(3)])

def corrupt(img,rng,only_hidden=False):
    im=Image.fromarray(img)
    if only_hidden: return np.asarray(hidden_corruption(im,rng))
    im=im.rotate(rng.uniform(-25,25),resample=Image.BILINEAR)
    a=np.asarray(im).astype(np.float32)
    a=a*rng.uniform(0.5,1.5)                                   # brightness
    m=a.mean(); a=(a-m)*rng.uniform(0.6,1.4)+m                 # contrast
    im=Image.fromarray(np.clip(a,0,255).astype(np.uint8))
    if rng.rand()<0.5: im=im.filter(ImageFilter.GaussianBlur(rng.uniform(0.5,1.5)))   # blur
    a=np.asarray(im).astype(np.float32)+rng.randn(32,32,3)*rng.uniform(0,15)          # noise
    a=np.clip(a,0,255).astype(np.uint8)
    h=rng.randint(8,25); w=min(32,int(round(256/h)))           # ~25% area
    y0,x0=rng.randint(0,33-h),rng.randint(0,33-w); a[y0:y0+h,x0:x0+w]=0              # occlusion
    return np.asarray(hidden_corruption(Image.fromarray(a),rng))

def build_shifted(X,seed,only_hidden=False):
    r=np.random.RandomState(seed); return np.stack([corrupt(x,r,only_hidden) for x in X])
Xsh=build_shifted(Xte,CORRUPT_SEED)
Xhid=build_shifted(Xte,CORRUPT_SEED+1,only_hidden=True)

mean,std=(0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)
def to_t(X):
    x=torch.tensor(X).permute(0,3,1,2).float()/255
    return (x-torch.tensor(mean).view(1,3,1,1))/torch.tensor(std).view(1,3,1,1)
Xva_t,yva_t=to_t(Xva),torch.tensor(yva); Xcl_t,yte_t=to_t(Xte),torch.tensor(yte)
Xsh_t,Xhid_t=to_t(Xsh),to_t(Xhid)
fig,ax=plt.subplots(2,8,figsize=(14,4))
for i in range(8):
    ax[0,i].imshow(Xte[i]); ax[1,i].imshow(Xsh[i]); ax[0,i].axis('off'); ax[1,i].axis('off')
ax[0,0].set_title('clean (top) vs shifted (bottom)',loc='left'); plt.show()

## Augmentation (applied on the fly, training data only)

In [ ]:
class RandomChannelSwap:
    def __init__(s,p): s.p=p
    def __call__(s,x): return x[torch.randperm(3)] if random.random()<s.p else x
class AddNoise:
    def __init__(s,sigma,p): s.sigma,s.p=sigma,p
    def __call__(s,x): return (x+torch.randn_like(x)*random.uniform(0,s.sigma)).clamp(0,1) if random.random()<s.p else x
def make_tf(cfg):
    t=[]
    if cfg.get('geo'): t+=[T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.RandomRotation(25)]
    if cfg.get('photo'): t+=[T.ColorJitter(0.5,0.5,0.4,0.05),T.RandomGrayscale(0.2),
                             T.RandomApply([T.GaussianBlur(3,(0.1,1.5))],p=0.3)]
    t+=[T.ToTensor()]
    if cfg.get('photo'): t+=[RandomChannelSwap(0.2),AddNoise(0.06,0.5)]
    t+=[T.Normalize(mean,std)]
    if cfg.get('cutout'): t+=[T.RandomErasing(p=0.6,scale=(0.1,0.3),ratio=(0.3,3.3),value=0)]
    return T.Compose(t)
class AugDS(Dataset):
    def __init__(s,X,y,tf): s.X,s.y,s.tf=X,y,tf
    def __len__(s): return len(s.X)
    def __getitem__(s,i): return s.tf(Image.fromarray(s.X[i])),int(s.y[i])

## Model: stem + two Inception modules + GAP + Dropout + FC

In [ ]:
def cbr(i,o,k): return nn.Sequential(nn.Conv2d(i,o,k,padding=k//2,bias=False),nn.BatchNorm2d(o),nn.ReLU(True))
class Inception(nn.Module):
    def __init__(s,i,b1,r3,b3,r5,b5,bp):
        super().__init__()
        s.b1=cbr(i,b1,1)
        s.b3=nn.Sequential(cbr(i,r3,1),cbr(r3,b3,3))
        s.b5=nn.Sequential(cbr(i,r5,1),cbr(r5,b5,5))
        s.bp=nn.Sequential(nn.MaxPool2d(3,1,1),cbr(i,bp,1))
    def forward(s,x): return torch.cat([s.b1(x),s.b3(x),s.b5(x),s.bp(x)],1)
class Net(nn.Module):
    def __init__(s,drop=0.3):
        super().__init__()
        s.features=nn.Sequential(cbr(3,32,3),cbr(32,64,3),nn.MaxPool2d(2),       # 16x16
                                 Inception(64,16,16,32,8,16,16),nn.MaxPool2d(2),  # 8x8, 80 ch
                                 Inception(80,32,32,64,16,32,32),nn.MaxPool2d(2)) # 4x4, 160 ch
        s.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(drop),nn.Linear(160,10))
    def forward(s,x): return s.head(s.features(x))
n_params=sum(p.numel() for p in Net().parameters() if p.requires_grad)
print('trainable params:',n_params); assert n_params<=1_000_000

## Training / evaluation utilities

In [ ]:
HP={'lr':3e-3,'wd':5e-4}
@torch.no_grad()
def predict(model,X,tta=False,bs=1000):
    model.eval(); out=[]
    for i in range(0,len(X),bs):
        xb=X[i:i+bs].to(dev)
        p=sum(F.softmax(model(v(xb)),1) for v in VIEWS)/len(VIEWS) if tta else F.softmax(model(xb),1)
        out.append(p.cpu())
    return torch.cat(out)
def acc(model,X,y,tta=False): return (predict(model,X,tta).argmax(1)==y).float().mean().item()*100
def shift(x,dx,dy):
    p=F.pad(x,(2,2,2,2),mode='replicate'); return p[:,:,2+dy:34+dy,2+dx:34+dx]
VIEWS=[lambda x:x,lambda x:x.flip(3),lambda x:shift(x,2,0),lambda x:shift(x,-2,0),
       lambda x:shift(x,0,2),lambda x:shift(x,0,-2)]

def train_one(cfg,seed,epochs=EPOCHS,bs=32,patience=25,hp=None):
    hp=hp or HP; set_seed(seed)
    g=torch.Generator(); g.manual_seed(seed)
    dl=DataLoader(AugDS(Xtr,ytr,make_tf(cfg)),batch_size=bs,shuffle=True,generator=g)
    model=Net().to(dev)
    opt=torch.optim.AdamW(model.parameters(),lr=hp['lr'],weight_decay=hp['wd'])
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=hp['lr'],total_steps=epochs*len(dl))
    crit=nn.CrossEntropyLoss(label_smoothing=cfg.get('ls',0.0))
    h={'train_loss':[],'train_acc':[],'val_acc':[]}; best,best_state,bad=-1,None,0
    for ep in range(epochs):
        model.train(); tl=tc=n=0
        for xb,yb in dl:
            xb,yb=xb.to(dev),yb.to(dev)
            if cfg.get('mixup'):
                lam=np.random.beta(cfg['mixup'],cfg['mixup']); perm=torch.randperm(len(xb),device=dev)
                out=model(lam*xb+(1-lam)*xb[perm]); loss=lam*crit(out,yb)+(1-lam)*crit(out,yb[perm])
            else:
                out=model(xb); loss=crit(out,yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tl+=loss.item()*len(xb); tc+=(out.argmax(1)==yb).sum().item(); n+=len(xb)
        va=acc(model,Xva_t,yva_t)
        h['train_loss'].append(tl/n); h['train_acc'].append(100*tc/n); h['val_acc'].append(va)
        if va>best: best,best_state,bad=va,copy.deepcopy(model.state_dict()),0
        else: bad+=1
        if bad>=patience: break
    model.load_state_dict(best_state); h['best_val']=best
    return model,h

## Hyperparameter selection (validation split only, seed 0, full augmentation recipe)
Test sets are NOT touched here.

In [ ]:
FULL={'geo':1,'photo':1,'cutout':1,'mixup':0.2,'ls':0.1}
sweep=[]
for lr in [1e-3,3e-3,1e-2]:
    for wd in [5e-4,5e-3]:
        _,h=train_one(FULL,0,hp={'lr':lr,'wd':wd}); sweep.append((lr,wd,h['best_val']))
sweep=pd.DataFrame(sweep,columns=['lr','wd','val_acc']); display(sweep)
b=sweep.loc[sweep.val_acc.idxmax()]; HP={'lr':b.lr,'wd':b.wd}; print('chosen',HP)

## Part A + D: baseline and ablation (3 seeds each)

In [ ]:
CONFIGS={
 'No augmentation':{},
 'Geometric only (crop, flip, rotation)':{'geo':1},
 'Photometric only (jitter, blur, noise)':{'photo':1},
 'Geometric + photometric':{'geo':1,'photo':1},
 '+ Cutout / Random Erasing':{'geo':1,'photo':1,'cutout':1},
 '+ Mixup + label smoothing':FULL,
}
RES={}
for name,cfg in CONFIGS.items():
    RES[name]=[]
    for s in SEEDS:
        m,h=train_one(cfg,s)
        RES[name].append(dict(seed=s,model=m,hist=h,val=h['best_val'],
                              clean=acc(m,Xcl_t,yte_t),shifted=acc(m,Xsh_t,yte_t)))
    print(name,[round(r['shifted'],1) for r in RES[name]])

In [ ]:
def ms(name,k): v=[r[k] for r in RES[name]]; return f'{np.mean(v):.2f} ± {np.std(v):.2f}'
cand=['Geometric + photometric','+ Cutout / Random Erasing','+ Mixup + label smoothing']
final_name=max(cand,key=lambda n:np.mean([r['val'] for r in RES[n]]))   # chosen by VALIDATION only
rows=[(n,ms(n,'val'),ms(n,'clean'),ms(n,'shifted')) for n in CONFIGS]
rows.append((f'Final model (= {final_name})',ms(final_name,'val'),ms(final_name,'clean'),ms(final_name,'shifted')))
table=pd.DataFrame(rows,columns=['Configuration','Val Acc','Clean Test','Shifted Test']); display(table)

### Part A: baseline report (no augmentation)

In [ ]:
base=RES['No augmentation']
print('Train acc (final epoch):',[round(r['hist']['train_acc'][-1],1) for r in base])
print('Val / Clean / Shifted :',ms('No augmentation','val'),'/',ms('No augmentation','clean'),'/',ms('No augmentation','shifted'))
fig,ax=plt.subplots(1,2,figsize=(11,4)); h=base[0]['hist']
ax[0].plot(h['train_loss'],label='train loss'); ax[0].set_title('Baseline loss (seed 0)'); ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend()
ax[1].plot(h['train_acc'],label='train acc'); ax[1].plot(h['val_acc'],label='val acc')
ax[1].set_title('Baseline accuracy (seed 0)'); ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy (%)'); ax[1].legend(); plt.show()

**Baseline diagnosis (fill in with your numbers):** The baseline reaches ~100% train accuracy but only modest validation/clean-test accuracy, so it **overfits** the 400 images (memorisation, huge train-val gap). Shifted-test accuracy falls much further below clean-test, which shows **covariate shift** (rotation, brightness, occlusion, noise, hidden colour change are unseen in training). So the failure is **both**: overfitting first, then poor robustness to distribution shift on top.

### Part B: augmentation questions
1. **Vertical flip hurts.** CIFAR-10 objects have a canonical upright orientation (sky on top, wheels below). The test set has ±25° rotations but never upside-down images, so flipped samples are unrealistic, waste capacity and blur class cues (e.g. horse, ship).
2. **±90° rotation vs ±25° test.** It covers the test range, but most training samples are in orientations never seen at test time. The model spends capacity on irrelevant poses, the task gets harder, and upright accuracy drops. Matching the augmentation range to the expected shift (about ±25–30°) is the best trade-off.
3. **Too much augmentation hurts a small model.** With about 80k parameters and 400 images, heavy augmentation (strong occlusion + mixup + darkening + rotation) can destroy the class evidence, so the model underfits. It also creates a mismatch between the training and test distributions, and BatchNorm statistics drift.

### Part C: regularisation placement
- **BatchNorm** after each conv, before ReLU. It stabilises activations, allows a higher LR and gives mild regularisation.
- **Dropout (0.3)** after GAP, before the final FC. Dense layers are where co-adaptation happens. Early conv features are spatially correlated and have few parameters, so dropout between them removes little redundancy and hurts learning.
- **Weight decay (AdamW, decoupled)** on all weights to keep them small.
- **LR schedule (OneCycle)** gives fast warm-up and annealing, which improves final convergence.
- **Early stopping** keeps the epoch with the best validation accuracy (validation split only).
- **Label smoothing** (0.1) stops overconfident predictions and pairs well with mixup.
- **GAP** instead of big FC layers removes most parameters.

### Part D: analysis (fill in from your table)
State which row gave the biggest jump in Shifted Test. Expected reasoning: **photometric augmentation** targets brightness/contrast, noise, blur and the hidden grayscale/channel-swap corruption. **Cutout** targets the 25% rectangular occlusion. **Geometric** targets the ±25° rotation. Quote the actual numbers from your table, and use the largest gain as your "single change that contributed most".

## Part E: Error analysis (final model, seed 0)

In [ ]:
fm=RES[final_name][0]['model']; P=predict(fm,Xsh_t).argmax(1).numpy()
cm=confusion_matrix(yte,P,normalize='true')
pc=cm.diagonal()*100
plt.figure(figsize=(8,4)); plt.bar(classes,pc); plt.xticks(rotation=45); plt.ylabel('Accuracy (%)'); plt.title('Per-class accuracy on shifted test'); plt.show()
plt.figure(figsize=(7,6)); plt.imshow(cm,cmap='Blues'); plt.colorbar(); plt.xticks(range(10),classes,rotation=45); plt.yticks(range(10),classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Normalised confusion matrix (shifted test)'); plt.show()
off=cm.copy(); np.fill_diagonal(off,0)
for i,j in sorted(np.ndindex(10,10),key=lambda t:-off[t])[:5]: print(f'{classes[i]} -> {classes[j]}: {off[i,j]:.2f}')
wrong=np.where(P!=yte)[0][:12]
fig,ax=plt.subplots(2,6,figsize=(14,5))
for a,i in zip(ax.ravel(),wrong):
    a.imshow(Xsh[i]); a.set_title(f'T:{classes[yte[i]]}\nP:{classes[P[i]]}',fontsize=9); a.axis('off')
plt.suptitle('Misclassified shifted images'); plt.show()

**Write-up (fill in):** (1) Name the most confused pairs from the printout (usually cat/dog, deer/horse, automobile/truck, bird/airplane) and say why: similar shape, texture and background, and low resolution (32×32). (2) For at least 8 of the images above, write one line each naming the cause (for example "ear region occluded by the black patch", "very dark image", "grayscale removed colour cue", "heavy rotation", "blur removed texture").

In [ ]:
hid=[acc(RES[final_name][i]['model'],Xhid_t,yte_t) for i in range(3)]
hid_base=[acc(RES['No augmentation'][i]['model'],Xhid_t,yte_t) for i in range(3)]
print(f'Hidden-corruption-only test | final: {np.mean(hid):.2f} ± {np.std(hid):.2f} | baseline: {np.mean(hid_base):.2f} ± {np.std(hid_base):.2f}')
print('Clean test (final):',ms(final_name,'clean'))

**Discussion (fill in):** Compare the hidden-only accuracy with clean-test. If it is close to clean-test, the photometric augmentation (jitter, random grayscale, channel swap) generalised to the hidden corruption. If it is much lower, the augmentation covered the listed corruptions but not this one.

## Part F (bonus): Test-time augmentation (6 views: identity, h-flip, 4 shifts of ±2 px)

In [ ]:
rs=[]
for i in range(3):
    m=RES[final_name][i]['model']
    t0=time.time(); a0=acc(m,Xsh_t,yte_t); t1=time.time(); a1=acc(m,Xsh_t,yte_t,tta=True); t2=time.time()
    rs.append((a0,a1,t1-t0,t2-t1))
rs=np.array(rs)
print(f'Shifted test: no TTA {rs[:,0].mean():.2f}  ->  TTA {rs[:,1].mean():.2f}  (delta {rs[:,1].mean()-rs[:,0].mean():+.2f})')
print(f'Inference time: {rs[:,2].mean():.2f}s -> {rs[:,3].mean():.2f}s  (about {len(VIEWS)}x the compute)')